# Advent of Code 2025

## Puzzle 1

### Part One

In [1]:
from dataclasses import dataclass
from pathlib import Path
from typing import Literal


@dataclass
class Move:
    dir: Literal["L", "R"]
    dist: int


def iter_moves():
    file = Path().parent / "inputs" / "01.txt"
    text = file.read_text()
    for line in text.splitlines():
        dir = line[0]
        dist = int(line[1:])
        assert dir == "L" or dir == "R"
        yield Move(dir, dist)


num_zeroes = 0
pos = 50
for move in iter_moves():
    if move.dir == "L":
        pos = (pos - move.dist) % 100
    elif move.dir == "R":
        pos = (pos + move.dist) % 100
    else:
        raise ValueError("Invalid move direction")
    if pos == 0:
        num_zeroes += 1

print(f"Number of times dial stopped on zero: {num_zeroes}")

Number of times dial stopped on zero: 1066


### Part Two

In [2]:
num_zero_clicks = 0
pos = 50
for move in iter_moves():
    if move.dir == "L":
        # => 100 not possible
        # 1..99 -> 0
        # -99..0 -> 1
        # -199..-100 -> 2
        num_zero_clicks += abs((pos - move.dist - 1) // 100)
        if pos == 0:
            num_zero_clicks -= 1
        pos = (pos - move.dist) % 100
    elif move.dir == "R":
        # <= 0 not possible
        # 1..99 -> 0
        # 100..199 -> 1
        # 200..299 -> 2
        num_zero_clicks += (pos + move.dist) // 100
        pos = (pos + move.dist) % 100
    else:
        raise ValueError("Invalid move direction")

print(f"Number of times dial passed zero: {num_zero_clicks}")

Number of times dial passed zero: 6223


## Puzzle 2

### Part One

In [3]:
from pathlib import Path


def iter_ranges():
    file = Path().parent / "inputs" / "02.txt"
    text = file.read_text()
    for line in text.split(","):
        start, end = line.split("-")
        yield int(start), int(end)

In [4]:
def is_invalid_id(id: int):
    num_digits = len(str(id))
    if num_digits % 2 != 0:
        # Only IDs with an even number of digits can be invalid
        return False
    # The first half of the ID must match the second half
    return str(id)[: num_digits // 2] == str(id)[num_digits // 2 :]

In [5]:
sum_invalid_ids = 0
for start, end in iter_ranges():
    for id in range(start, end + 1):
        if is_invalid_id(id):
            sum_invalid_ids += id

print(sum_invalid_ids)

28846518423


### Part Two

In [6]:
def is_invalid_id(id: int):
    num_digits = len(str(id))  # 4
    for i in range(num_digits - 1):  # 0..2
        str_to_repeat = str(id)[: i + 1]  # 1..3
        if num_digits % len(str_to_repeat) != 0:
            continue
        num_times_to_repeat = num_digits // len(str_to_repeat)
        if str(id) == str_to_repeat * num_times_to_repeat:
            return True
    return False

In [7]:
sum_invalid_ids = 0
for start, end in iter_ranges():
    for id in range(start, end + 1):
        if is_invalid_id(id):
            sum_invalid_ids += id

print(sum_invalid_ids)

31578210022


## Puzzle 3

### Part One

In [8]:
from pathlib import Path


def iter_banks():
    file = Path().parent / "inputs" / "03.txt"
    text = file.read_text()
    for line in text.splitlines():
        yield [int(c) for c in line]


In [9]:
def get_maximum_joltage(bank: list[int]):
    if len(bank) < 2:
        return 0

    # Find the largest battery by scanning the bank from left to right,
    # keeping the leftmost battery in case of a tie,
    # and excluding the very last battery (since we need a second battery)
    first_battery_idx = None
    first_battery_joltage = None
    for i, battery in enumerate(bank[:-1]):
        if first_battery_joltage is None or battery > first_battery_joltage:
            first_battery_idx = i
            first_battery_joltage = battery
            if battery == 9:
                break  # can't find a better battery

    assert first_battery_idx is not None
    assert first_battery_joltage is not None

    # Second battery joltage is the largest battery to the right of the
    # first battery
    second_battery_joltage = max(bank[first_battery_idx + 1 :])

    # Calculate the combined joltage
    return int(str(first_battery_joltage) + str(second_battery_joltage))

In [10]:
total_joltage = 0
for bank in iter_banks():
    total_joltage += get_maximum_joltage(bank)
print(total_joltage)

17142


### Part Two

In [11]:
def get_maximum_joltage(bank: list[int], n=12):
    if len(bank) < n:
        raise ValueError("Not enough batteries in bank")
    if n < 1:
        raise ValueError("Need to choose at least one battery")
    # If we can only choose one battery, we take the largest one
    if n == 1:
        return max(bank)

    # Find the largest battery by scanning the bank from left to right,
    # keeping the leftmost battery in case of a tie,
    # and excluding the last n-1 batteries (since we need n-1 more batteries
    # after this one)
    next_battery_idx = None
    next_battery_joltage = None
    for i, battery in enumerate(bank[: -(n - 1)]):
        if next_battery_joltage is None or battery > next_battery_joltage:
            next_battery_idx = i
            next_battery_joltage = battery
            if battery == 9:
                break  # can't find a better battery

    assert next_battery_idx is not None
    assert next_battery_joltage is not None

    return int(
        str(next_battery_joltage)
        + str(get_maximum_joltage(bank[next_battery_idx + 1 :], n=n - 1))
    )

In [12]:
total_joltage = 0
for bank in iter_banks():
    total_joltage += get_maximum_joltage(bank)
print(total_joltage)

169935154100102


## Puzzle 4

### Part One

In [13]:
from pathlib import Path


def get_grid():
    file = Path().parent / "inputs" / "04.txt"
    text = file.read_text()
    return list(text.splitlines())

In [14]:
from enum import Enum
from typing import Sequence


class Direction(Enum):
    RIGHT = (1, 0)
    DOWN_RIGHT = (1, 1)
    DOWN = (0, 1)
    DOWN_LEFT = (-1, 1)
    LEFT = (-1, 0)
    UP_LEFT = (-1, -1)
    UP = (0, -1)
    UP_RIGHT = (1, -1)


def can_access(grid: Sequence[Sequence[str]], *, x: int, y: int):
    num_adjacent = 0
    for dir in Direction:
        check_x = x + dir.value[0]
        check_y = y + dir.value[1]
        if check_y not in range(len(grid)):
            continue  # out of bounds
        if check_x not in range(len(grid[check_y])):
            continue  # out of bounds
        cell = grid[check_y][check_x]
        if cell == "@":
            num_adjacent += 1

    return num_adjacent < 4

In [15]:
num_accessible = 0
grid = get_grid()
for y in range(len(grid)):
    for x in range(len(grid[y])):
        cell = grid[y][x]
        if cell == "@" and can_access(grid, x=x, y=y):
            num_accessible += 1

print(num_accessible)

1372


### Part Two

In [16]:
from typing import Sequence


def find_accessible(grid: Sequence[Sequence[str]]):
    accessible = []
    for y in range(len(grid)):
        for x in range(len(grid[y])):
            cell = grid[y][x]
            if cell == "@" and can_access(grid, x=x, y=y):
                accessible.append((x, y))
    return accessible


# Get a mutable copy of the grid
mgrid = []
for row in get_grid():
    # Representing rows as lists of single-character strings
    # instead of lists of (multi-character) strings
    mgrid.append([c for c in row])

# Remove all accessible rolls until there are no more accessible rolls
num_removed = 0
accessible = find_accessible(mgrid)
while len(accessible) > 0:
    for x, y in accessible:
        mgrid[y][x] = "."
        num_removed += 1
    accessible = find_accessible(mgrid)

print(num_removed)

7922


## Puzzle 5

### Part One

In [17]:
from pathlib import Path

def get_ranges_and_ingredients():
    file = Path().parent / "inputs" / "05.txt"
    text = file.read_text()

    ranges_text, ingredients_text = text.split("\n\n")

    ranges = []
    for line in ranges_text.splitlines():
        start, end = line.split('-')
        ranges.append([int(start), int(end)])

    ingredients = []
    for line in ingredients_text.splitlines():
        ingredients.append(int(line))

    return ranges, ingredients


In [18]:
ranges, ingredients = get_ranges_and_ingredients()

num_fresh = 0
for ingredient in ingredients:
    for start, end in ranges:
        if ingredient in range(start, end+1):
            num_fresh += 1
            break

print(num_fresh)

601


### Part Two

In [19]:
class Range:
    """Inclusive range of integers."""

    start: int
    end: int

    def __init__(self, start: int, end: int):
        if start > end:
            raise ValueError()

        self.start = start
        self.end = end

    def __repr__(self) -> str:
        return f"Range({self.start}, {self.end})"

    def touches(self, other: "Range") -> bool:
        """Checks whether this range touches another range."""
        return not (self.end < other.start - 1 or self.start > other.end + 1)

    def merge(self, other: "Range") -> "Range":
        """Creates a new range by merging this range with another. The ranges
        MUST be touching."""
        return Range(start=min(self.start, other.start), end=max(self.end, other.end))

In [20]:
def get_ranges():
    file = Path().parent / "inputs" / "05.txt"
    text = file.read_text()

    ranges_text, _ = text.split("\n\n")

    ranges: list[Range] = []
    for line in ranges_text.splitlines():
        start, end = line.split('-')
        ranges.append(Range(int(start), int(end)))

    return ranges

In [ ]:
ranges = get_ranges()

# Prepare for merging by sorting by starting number
# This way we only need to do a linear scan of all ranges
ranges.sort(key=lambda r: r.start)

# Merge any ranges that touch
i = 0
while i < len(ranges) - 1:
    if ranges[i].touches(ranges[i + 1]):
        merged = ranges[i].merge(ranges[i + 1])
        ranges.pop(i)
        ranges.pop(i)  # this was ranges[i + 1] before
        ranges.insert(i, merged)
    else:
        i += 1

In [22]:
# Sanity check, no ranges should be touching after merging
for i in range(len(ranges)):
    for j in range(i+1, len(ranges)):
        assert not ranges[i].touches(ranges[j])

In [23]:
# Sum up the number of elements in every range
num_fresh_ingredients = sum(r.end - r.start + 1 for r in ranges)
num_fresh_ingredients

367899984917516